# Lab - Structured Tool Errors in Code

Three tasks on what a tool should return when things go wrong. A tool that fails silently,
or crashes the loop, leaves both Claude and the customer without an answer.

| Task | What you learn |
| --- | --- |
| 1 | An order that does not exist is a normal answer - not found is data, not an error |
| 2 | A real failure is a structured error: a flag, a category, a retryable flag, two messages |
| 3 | The tool_result carries is_error, and Claude explains in the customer's words |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# The policy moved to the tool: Claude must not judge the return window itself.
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Use the tools to look up orders and to start returns. "
    "The start_return tool checks the return policy; do not "
    "judge it yourself. Answer in one or two sentences."
)

# A second order: a lamp, delivered forty-five days ago.
ORDERS = {
    "ORD-12345678": {
        "item": "jacket",
        "status": "delivered",
        "delivered_days_ago": 20,
    },
    "ORD-87654321": {
        "item": "lamp",
        "status": "delivered",
        "delivered_days_ago": 45,
    },
}

TOOLS = [
    {
        "name": "lookup_order",
        "description": "Look up an order by its ID. Returns the "
                       "item, the status and how many days ago "
                       "it was delivered.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "start_return",
        "description": "Start a return for a delivered order that "
                       "is inside the 30-day window. Returns the "
                       "return label number.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
]

---

## Task 1 - Not found is data, not an error

In [ ]:
# ============================================================
# TASK 1 - Not found is data, not an error
# ============================================================
#
# WHAT TO DO
#   Finish lookup_order(): when the order does not exist,
#   return found False and a short message; when it exists,
#   return found True and the order.
#
# WHY IT MATTERS
#   A customer can give an order number that does not exist.
#   That is not an error - it is a normal answer, just an empty
#   one. Raising here would crash the loop; flagging it as an
#   error would tell Claude the tool is broken. found False
#   tells Claude exactly what happened: there is no such order.
#
# WHERE TO SEE IT IN THE LECTURE
#   "When the order does not exist, that is not an error".
#
# HOW TO DO IT
#   Three blanks:
#
#       None    what ORDERS.get() gives for an unknown ID
#       False   the order was not found
#       True    the order was found
# ============================================================

# TODO: replace each ... below with the value named beside it
def lookup_order(order_id):
    order = ORDERS.get(order_id)
    if order is ...:   # None - ORDERS.get() found nothing
        return {"found": ..., "message": "No order with this ID."}   # False
    return {"found": ..., "order": order}   # True

print(lookup_order("ORD-99999999"))
print(lookup_order("ORD-87654321"))

check("not_found", lookup_order=lookup_order)

---

## Task 2 - A structured error when the policy blocks it

In [ ]:
# ============================================================
# TASK 2 - A structured error when the policy blocks it
# ============================================================
#
# WHAT TO DO
#   Finish start_return(): an order past the 30-day window
#   is refused with a structured error. Inside the window, it
#   returns the label as before.
#
# WHY IT MATTERS
#   Here the policy blocks the action - a real failure. A free
#   text error leaves Claude guessing; an exception crashes the
#   loop. A structured error tells Claude three things at once:
#   that the call failed, why, and whether trying again makes
#   sense. Two messages come back as plain data: one that is
#   safe to show the customer, and one for the logs.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The start return tool is different".
#
# HOW TO DO IT
#   Three blanks:
#
#       True         the call failed
#       "business"   the category: a policy blocked it
#       False        retrying cannot change the delivery date
# ============================================================

# TODO: replace each ... below with the value named beside it
def start_return(order_id):
    order = ORDERS.get(order_id)
    if order["delivered_days_ago"] > 30:
        return {
            "isError": ...,         # True - the call failed
            "errorCategory": ...,   # "business" - the policy blocked it
            "isRetryable": ...,     # False - retrying cannot help
            "customerMessage": "This order is past the 30-day "
                               "return window.",
            "developerMessage": "delivered_days_ago="
                                f"{order['delivered_days_ago']} > 30",
        }
    return {"return_label": "RET-2048", "order_id": order_id}

print(start_return("ORD-87654321"))
print(start_return("ORD-12345678"))

check("business_error", start_return=start_return)

---

## Task 3 - Flag the tool_result, and let Claude explain

In [ ]:
# ============================================================
# TASK 3 - Flag the tool_result, and let Claude explain
# ============================================================
#
# WHAT TO DO
#   The loop from the last lab is now a function, handle(),
#   for one customer message. Set the is_error flag on each
#   tool_result from the error flag in the output, then run
#   the two cases.
#
# WHY IT MATTERS
#   is_error is the flag the protocol understands: it tells
#   Claude this call failed. The unknown order is not flagged -
#   Claude simply asks the customer to check the number. The
#   lamp is: Claude tells the customer the window has passed,
#   in the words of the customer message. The developer
#   message never reaches the customer.
#
# WHERE TO SEE IT IN THE LECTURE
#   "In the tool result, we also set the is error flag".
#
# HOW TO DO IT
#   Three blanks:
#
#       "tool_use"   keep looping while Claude asks for tools
#       output       the tool's result, sent as JSON
#       "isError"    the flag in the output that marks an error
# ============================================================

TOOL_FUNCTIONS = {"lookup_order": lookup_order,
                  "start_return": start_return}

CASES = [
    "Can I return my order ORD-99999999?",
    "I want to return the lamp from order ORD-87654321.",
]


# TODO: replace each ... below with the value named beside it
def handle(text):
    messages = [{"role": "user", "content": text}]
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=500,
            output_config={"effort": "low"},
            system=SYSTEM,
            tools=TOOLS,
            messages=messages,
        )
        if response.stop_reason != ...:   # "tool_use"
            break
        messages.append({"role": "assistant",
                         "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = TOOL_FUNCTIONS[block.name](**block.input)
                print(f"  {block.name} -> {output}")
                results.append({"type": "tool_result",
                                "tool_use_id": block.id,
                                "content": json.dumps(...),   # output
                                "is_error": bool(output.get(...))})   # "isError"
        messages.append({"role": "user", "content": results})
    for block in response.content:
        if block.type == "text":
            print("ShopAssist:", block.text)
    return response, messages   # provided: so the check can read the conversation

try:
    cases = []
    for text in CASES:
        print("Customer:", text)
        cases.append(handle(text))
        print()
except TypeError as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("is_error", cases=cases)

---

## Done

Not found is data, not an error. A real failure is a structured error with a flag, a
category, a retryable flag, and two messages: one for the customer, one for the logs.
The `tool_result` carries `is_error`, and Claude answers in the words of the customer
message.